<a href="https://colab.research.google.com/github/CesarChaMal/ollama/blob/main/ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# Download and run the Ollama Linux install script
!curl https://ollama.ai/install.sh | sh
!command -v systemctl >/dev/null && sudo systemctl stop ollama

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0>>> Downloading ollama...
100  8354    0  8354    0     0  35088      0 --:--:-- --:--:-- --:--:-- 35100
############################################################################################# 100.0%
>>> Installing ollama to /usr/local/bin...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 0.0.0.0:11434.
>>> Install complete. Run "ollama" from the command line.
System has not been booted with systemd as init system (PID 1). Can't operate.
Failed to connect to bus: Host is down


In [ ]:
!rm -rf *
!git init .
!git remote add origin https://github.com/ollama-webui/ollama-webui.git
!git pull origin main
!#git clone https://github.com/ollama-webui/ollama-webui.git .

# Copying required .env file
!cp -RPp example.env .env

# Building Frontend Using Node
!#npm install --save-dev @tsconfig/svelte
!node svelte.config.js
!npm install
!npm run build

# or Building Frontend Using Bun
# bun install
# bun run build

# Serving Frontend with the Backend
!pip install -r backend/requirements.txt -U
!sh backend/start.sh

!pip install aiohttp pyngrok

import os
import asyncio
from aiohttp import ClientSession
from pyngrok import ngrok

# Set LD_LIBRARY_PATH so the system NVIDIA library becomes preferred
# over the built-in library. This is particularly important for
# Google Colab which installs older drivers
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

# Set up ngrok
ngrok.set_auth_token("")
public_url = ngrok.connect(3000, "http")
print("Public URL:", public_url)

# Set the OLLAMA_HOST environment variable
os.environ['OLLAMA_HOST'] = public_url.public_url


async def run(cmd):
    '''
    run is a helper function to run subcommands asynchronously.
    '''
    print('>>> starting', *cmd)
    p = await asyncio.subprocess.create_subprocess_exec(
        *cmd,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )

    async def pipe(lines):
        async for line in lines:
            print(line.strip().decode('utf-8'))

    await asyncio.gather(
        pipe(p.stdout),
        pipe(p.stderr),
    )


await asyncio.gather(
    run(['ollama', 'serve']),
    run(['ngrok', 'http', '--log', 'stderr', '11434']),
)


async def run_ollama_commands():
    # Define the commands you want to run
    commands = [
        ['ollama', 'list'],
        ['ollama', 'run', 'mistral']
    ]

    for cmd in commands:
        print('>>> Executing:', ' '.join(cmd))
        process = await asyncio.create_subprocess_exec(*cmd, stdout=asyncio.subprocess.PIPE,
                                                       stderr=asyncio.subprocess.PIPE)

        # Capture the output
        stdout, stderr = await process.communicate()
        if stdout:
            print(stdout.decode())
        if stderr:
            print(stderr.decode())


# Run the ollama commands
await run_ollama_commands()


The previous cell starts two processes, `ollama` and `ngrok`. The log output will show a line like the following which describes the external address.

```
t=2023-11-12T22:55:56+0000 lvl=info msg="started tunnel" obj=tunnels name=command_line addr=http://localhost:11434 url=https://8249-34-125-179-11.ngrok.io
```

The external address in this case is `https://8249-34-125-179-11.ngrok.io` which can be passed into `OLLAMA_HOST` to access this instance.

```bash
export OLLAMA_HOST=https://8249-34-125-179-11.ngrok.io
ollama list
ollama run mistral
```

In [ ]:
!export OLLAMA_HOST=https://8249-34-125-179-11.ngrok.io
!ollama list
!ollama run mistral